### Forward Model — Soft + Hard Combined Data

Train the same `ImprovedRegressionModel` on the **unaugmented** soft+hard combined dataset prepared by `prepare_soft_hard_combined.ipynb`.

#### Data & Scalers
- Train: `data/combined_soft_hard_train.xlsx` (16,016 rows)
- Val: `data/combined_soft_hard_val.xlsx` (4,095 rows)
- Scalers: `artifacts/high_stiffness/scaler_x_soft_hard.pkl`, `artifacts/high_stiffness/scaler_y_soft_hard.pkl`
  (fitted only on the combined training set)

#### Model columns (selected by name, ID columns excluded)
- Inputs (`X`): `Ep1`, `Ep2`, `Ep3`, `Epp1`, `Epp2`, `Epp3`, `t1`, `t2`
- Outputs (`y`): `EpH`, `EppH`

#### Outputs (new filenames under `data/`, do not overwrite previous artifacts)
- `artifacts/high_stiffness/forward_model_weights_soft_hard.pt`
- `artifacts/high_stiffness/forward_model_training_history_soft_hard.csv`
- `artifacts/high_stiffness/forward_model_predictions_soft_hard.pkl`

Architecture, DWA training, and plotting style match `forward model_new.ipynb`.


### Environment Settings

In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import joblib
from matplotlib.ticker import MultipleLocator
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

In [ ]:
# Use serif + bold global settings before drawing
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif']  = ['DejaVu Serif', 'Liberation Serif']
plt.rcParams['font.weight'] = 'bold'

### 1. Define the forward model


In [ ]:
class ImprovedRegressionModel(nn.Module):
    def __init__(self,
                 moduli_dim: int = 6,
                 dist_dim: int = 2,
                 hidden_dim: int = 256,
                 trunk_layers: int = 2,
                 dropout: float = 0.15,
                 trunk_dropout: float = 0.20):  # Stronger regularization on the conditional (trunk) branch
        """
        The input x is composed of two parts:
          - x_mod (batch, 6): Fixed 6-column material moduli
          - x_dist (batch, 2): Distribution values predicted by the diffusion model
        The outputs y0, y1 are two effective moduli.
        """
        super().__init__()
        assert dist_dim == 2, "This model expects dist_dim=2"
        self.moduli_dim = moduli_dim
        self.dist_dim   = dist_dim

        # --- 1) Trunk for processing 6-dimensional moduli ---
        trunk = []
        in_dim = moduli_dim
        for _ in range(trunk_layers):
            trunk.append(nn.Linear(in_dim, hidden_dim))
            trunk.append(nn.LayerNorm(hidden_dim))
            trunk.append(nn.GELU())
            trunk.append(nn.Dropout(trunk_dropout))  # Stronger regularization on conditional features
            in_dim = hidden_dim
        self.trunk = nn.Sequential(*trunk)  # output (batch, hidden_dim)

        # --- 2) Simple mapping for processing distribution x_dist ---
        #    Directly map (batch,2) to (batch, hidden_dim//4), then concatenate to trunk output
        self.dist_embed = nn.Sequential(
            nn.Linear(dist_dim, hidden_dim // 4),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        # --- 3) Fusion dimension after concatenating trunk and dist_embed outputs ---
        fusion_dim = hidden_dim + (hidden_dim // 4)
        self.fusion_norm = nn.LayerNorm(fusion_dim)

        # --- 4) Head for storage modulus (y0) ---
        self.head0 = nn.Sequential(
            nn.Linear(fusion_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1)
        )

        # --- 5) Head for loss modulus (y1): deeper network ---
        self.head1 = nn.Sequential(
            nn.Linear(fusion_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, hidden_dim // 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 4, 1)
        )

        # --- 6) Learn variance parameters (optional for NLL loss) ---
        self.log_sigma0 = nn.Parameter(torch.zeros(()))
        self.log_sigma1 = nn.Parameter(torch.zeros(()))

        # --- 7) Initialize all linear layer weights ---
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (batch, 8) = [ x_mod (6 dims) | x_dist (2 dims) ]
        Returns (batch, 2): first column y0, second column y1
        """
        # Split
        x_mod  = x[:, : self.moduli_dim]  # (batch, 6)
        x_dist = x[:, self.moduli_dim:]   # (batch, 2)

        # 1) First, send moduli into the trunk
        z_mod = self.trunk(x_mod)         # (batch, hidden_dim)

        # 2) Send distribution into dist_embed
        z_dist = self.dist_embed(x_dist)  # (batch, hidden_dim//4)

        # 3) Concatenate features + LayerNorm
        z = torch.cat([z_mod, z_dist], dim=1)  # (batch, hidden_dim + hidden_dim//4)
        z = self.fusion_norm(z)

        # 4) Pass through two heads to get y0, y1
        y0 = self.head0(z)  # (batch, 1)
        y1 = self.head1(z)  # (batch, 1)

        return torch.cat([y0, y1], dim=-1)  # (batch, 2)


### 2. Data preparation: Load soft+hard combined data and scalers


In [ ]:
# 1) Load pre-split soft+hard combined data
train_path = "data/combined_soft_hard_train.xlsx"
val_path = "data/combined_soft_hard_val.xlsx"

X_cols = ["Ep1", "Ep2", "Ep3", "Epp1", "Epp2", "Epp3", "t1", "t2"]
y_cols = ["EpH", "EppH"]

print("Loading soft+hard combined data...")
df_train = pd.read_excel(train_path, header=0)
df_val = pd.read_excel(val_path, header=0)

# Select model columns by name (exclude NewGroupID / HardSetID / etc.)
missing_tr = [c for c in X_cols + y_cols if c not in df_train.columns]
missing_va = [c for c in X_cols + y_cols if c not in df_val.columns]
if missing_tr or missing_va:
    raise ValueError(f"Missing columns — train: {missing_tr}, val: {missing_va}")

X_tr = df_train[X_cols].to_numpy(dtype=float)   # (N_train, 8)
Y_tr = df_train[y_cols].to_numpy(dtype=float)   # (N_train, 2)
X_val = df_val[X_cols].to_numpy(dtype=float)    # (N_val, 8)
Y_val = df_val[y_cols].to_numpy(dtype=float)    # (N_val, 2)

print(f"Train set: {X_tr.shape[0]} samples, X={X_tr.shape}, y={Y_tr.shape}")
print(f"Validation set: {X_val.shape[0]} samples, X={X_val.shape}, y={Y_val.shape}")
if "MaterialType" in df_train.columns:
    print("Train MaterialType counts:")
    print(df_train["MaterialType"].value_counts().to_string())
if "MaterialType" in df_val.columns:
    print("Val MaterialType counts:")
    print(df_val["MaterialType"].value_counts().to_string())

# 2) Load pre-fitted soft+hard scalers (fitted on combined training set only)
scaler_x_path = "artifacts/high_stiffness/scaler_x_soft_hard.pkl"
scaler_y_path = "artifacts/high_stiffness/scaler_y_soft_hard.pkl"

print(f"\nLoading pre-fitted soft+hard scalers...")
scaler_x = joblib.load(scaler_x_path)
scaler_y = joblib.load(scaler_y_path)

print(f"Loaded scaler_x from: {scaler_x_path}")
print(f"Loaded scaler_y from: {scaler_y_path}")

print(f"\nScaler parameters (first 3 features):")
print(f"  X mean: {scaler_x.mean_[:3]}")
print(f"  X std:  {scaler_x.scale_[:3]}")
print(f"  Y mean: {scaler_y.mean_}")
print(f"  Y std:  {scaler_y.scale_}")

# 3) Transform both splits using pre-fitted scalers
X_tr_scaled = scaler_x.transform(X_tr)
Y_tr_scaled = scaler_y.transform(Y_tr)
X_val_scaled = scaler_x.transform(X_val)
Y_val_scaled = scaler_y.transform(Y_val)

# 4) Convert to tensors & DataLoaders
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nUsing device: {device}")

train_ds = TensorDataset(
    torch.tensor(X_tr_scaled, dtype=torch.float32, device=device),
    torch.tensor(Y_tr_scaled, dtype=torch.float32, device=device)
)
val_ds = TensorDataset(
    torch.tensor(X_val_scaled, dtype=torch.float32, device=device),
    torch.tensor(Y_val_scaled, dtype=torch.float32, device=device)
)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=128, shuffle=False)

print(f"\nData preparation completed!")
print(f"✓ Soft+hard combined train/val loaded by column name")
print(f"✓ Using soft+hard pre-fitted scalers (no data leakage)")


In [ ]:
def eval_loader_losses(model, loader, criterion, device):
    """
    Returns the MSE for each head on the loader (with dropout turned off).
    """
    model.eval()  # Ensure Dropout/LayerNorm is in eval mode
    total0 = total1 = count = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            pred   = model(xb)           # (batch, 2)
            l0 = criterion(pred[:, 0], yb[:, 0])
            l1 = criterion(pred[:, 1], yb[:, 1])
            b  = xb.size(0)
            total0 += l0.item() * b
            total1 += l1.item() * b
            count  += b
    return total0 / count, total1 / count

def eval_loader_loss(model, loader, criterion, device):
    """
    Returns the overall MSE on the loader (combined MSE of two heads, with dropout turned off).
    """
    model.eval()
    total = count = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            pred   = model(xb)  # (batch, 2)
            total += criterion(pred, yb).item() * xb.size(0)
            count += xb.size(0)
    return total / count

### 3. Soft+hard forward model training


In [ ]:
# Hyperparameters, model, optimizer, and scheduler - CONSERVATIVE IMPROVEMENT
epochs = 100  # Keep original number of training epochs
temp   = 0.5
dwa_warmup_epochs = 5
storage_bias = 1.2
loss_bias = 0.8
cond_noise_std = 0.02  # Noise augmentation for conditional inputs
trunk_dropout = 0.20

# Conservative improved model configuration
forward_model = ImprovedRegressionModel(
    moduli_dim=6,
    dist_dim=2,
    hidden_dim=256,
    trunk_layers=2,
    dropout=0.15,
    trunk_dropout=trunk_dropout
)
forward_model.to(device)

# Conservative improved optimizer configuration
optimizer = optim.AdamW(forward_model.parameters(), lr=1e-4, weight_decay=4e-5)  # Further increase weight decay
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)
criterion = nn.MSELoss()

# Conservative improved configuration
print(f"Soft+Hard Forward Model Training Configuration:")
print(f"  Epochs: {epochs}")
print(f"  Learning rate: 1e-4 (unchanged)")
print(f"  Weight decay: 4e-5 (moderate increase from 1e-5)")
print(f"  Dropout(head/dist): 0.15")
print(f"  Trunk dropout: {trunk_dropout}")
print(f"  Cond input noise std: {cond_noise_std}")
print(f"  DWA warmup epochs: {dwa_warmup_epochs}")
print(f"  DWA bias (storage, loss): ({storage_bias}, {loss_bias})")
print(f"  Hidden dim: 256 (unchanged)")
print(f"  DWA temperature: {temp}")
print(f"  No early stopping (keep original behavior)")

# DWA & record curves
train_loss0, train_loss1     = [], []
val_loss0,   val_loss1       = [], []

# Record MSE for each head in 'eval mode'
train_eval0, train_eval1     = [], []
val_eval0,   val_eval1       = [], []

# (If you want to keep the original overall eval curve)
train_eval_losses, val_eval_losses = [], []

# === Data structures for saving training history ===
training_history = {
    'epoch': [],
    'train_loss0': [],
    'train_loss1': [],
    'val_loss0': [],
    'val_loss1': [],
    'train_eval0': [],
    'train_eval1': [],
    'val_eval0': [],
    'val_eval1': [],
    'train_eval_combined': [],
    'val_eval_combined': [],
    'learning_rate': [],
    'dwa_weight0': [],
    'dwa_weight1': []
}

for epoch in range(epochs):
    # —— 1) Calculate DWA weights w0, w1 ——
    if epoch < dwa_warmup_epochs:
        raw_w0 = storage_bias
        raw_w1 = loss_bias
    elif len(train_loss0) >= 2 and len(train_loss1) >= 2:
        r0 = train_loss0[-1] / max(train_loss0[-2], 1e-8)
        r1 = train_loss1[-1] / max(train_loss1[-2], 1e-8)
        raw_w0 = np.exp(r0 / temp)
        raw_w1 = np.exp(r1 / temp)
    else:
        raw_w0 = raw_w1 = 1.0
    s = raw_w0 + raw_w1
    w0 = raw_w0 / s
    w1 = raw_w1 / s

    # —— 2) Training phase ——
    forward_model.train()  # Dropout 
    sum0 = sum1 = count = 0

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)

        if cond_noise_std > 0:
            noise = torch.randn_like(xb[:, :forward_model.moduli_dim]) * cond_noise_std
            xb_aug = xb.clone()
            xb_aug[:, :forward_model.moduli_dim] += noise
        else:
            xb_aug = xb

        pred = forward_model(xb_aug)  # (batch, 2)

        loss0 = criterion(pred[:, 0], yb[:, 0])
        loss1 = criterion(pred[:, 1], yb[:, 1])

        # Use DWA weights instead of learned sigma
        loss = w0 * loss0 + w1 * loss1

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(forward_model.parameters(), max_norm=1.0)
        optimizer.step()

        b = xb.size(0)
        sum0  += loss0.item() * b
        sum1  += loss1.item() * b
        count += b

    train0 = sum0 / count
    train1 = sum1 / count
    train_loss0.append(train0)
    train_loss1.append(train1)

    # —— 3) Validation phase ——
    forward_model.eval()  # trun off Dropout
    v0 = v1 = count = 0

    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = forward_model(xb)
            l0   = criterion(pred[:, 0], yb[:, 0]).item()
            l1   = criterion(pred[:, 1], yb[:, 1]).item()
            b    = xb.size(0)
            v0  += l0 * b
            v1  += l1 * b
            count += b

    val0 = v0 / count
    val1 = v1 / count
    val_loss0.append(val0)
    val_loss1.append(val1)

    # —— 4) eval_loader_losses records MSE for each head on train/val ——
    train_l0_eval, train_l1_eval = eval_loader_losses(
        forward_model, train_loader, criterion, device
    )
    val_l0_eval,   val_l1_eval   = eval_loader_losses(
        forward_model, val_loader,   criterion, device
    )
    train_eval0.append(train_l0_eval)
    train_eval1.append(train_l1_eval)
    val_eval0  .append(val_l0_eval)
    val_eval1  .append(val_l1_eval)

    # —— 5) eval_loader_loss records overall train/val MSE ——
    overall_train_eval = eval_loader_loss(
        forward_model, train_loader, criterion, device
    )
    overall_val_eval   = eval_loader_loss(
        forward_model, val_loader,   criterion, device
    )
    train_eval_losses.append(overall_train_eval)
    val_eval_losses  .append(overall_val_eval)

    # —— 6) Print metrics ——
    ratio = (epoch + 1) / epochs
    print(
        f"Epoch {epoch+1}/{epochs} ({ratio:.2%})  "
        f"w0={w0:.3f}, w1={w1:.3f}  |  "
        f"Train(storage)={train0:.4e}, Train(loss)={train1:.4e}  |  "
        f"Val(storage)={val_loss0[-1]:.4e}, Val(loss)={val_loss1[-1]:.4e}  |  "
        f"TrainEval(storage)={train_l0_eval:.4e}, TrainEval(loss)={train_l1_eval:.4e}  |  "
        f"ValEval(storage)={val_l0_eval:.4e}, ValEval(loss)={val_l1_eval:.4e}  |  "
        f"OverallTrainEval={overall_train_eval:.4e}, OverallValEval={overall_val_eval:.4e}"
    )

    # —— 7) Learning rate scheduling ——
    scheduler.step()
    
    # —— 8) Record training history ——
    lr_current = optimizer.param_groups[0]['lr']
    training_history['epoch'].append(epoch + 1)
    training_history['train_loss0'].append(train0)
    training_history['train_loss1'].append(train1)
    training_history['val_loss0'].append(val0)
    training_history['val_loss1'].append(val1)
    training_history['train_eval0'].append(train_l0_eval)
    training_history['train_eval1'].append(train_l1_eval)
    training_history['val_eval0'].append(val_l0_eval)
    training_history['val_eval1'].append(val_l1_eval)
    training_history['train_eval_combined'].append(overall_train_eval)
    training_history['val_eval_combined'].append(overall_val_eval)
    training_history['learning_rate'].append(lr_current)
    training_history['dwa_weight0'].append(w0)
    training_history['dwa_weight1'].append(w1)

# save model & history under data/ with new soft_hard filenames
results_dir = Path("data")
results_dir.mkdir(parents=True, exist_ok=True)

weights_path = results_dir / "forward_model_weights_soft_hard.pt"
history_path = results_dir / "forward_model_training_history_soft_hard.csv"

torch.save(forward_model.state_dict(), weights_path)
print(f"\nSoft+hard model saved to: {weights_path}")

history_df = pd.DataFrame(training_history)
history_df.to_csv(history_path, index=False)
print(f"Soft+hard training history saved to: {history_path}")


### 4. Generate predictions and save data for scatter plots


In [ ]:
# Use validation set for model inference
# Note: X_val_scaled and Y_val_scaled are already transformed using pre-fitted scalers

# Convert scaled validation data to tensors
x_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32, device=device)
y_val_tensor = torch.tensor(Y_val_scaled, dtype=torch.float32, device=device)

# Model inference
forward_model.eval()
with torch.no_grad():
    preds_std = forward_model(x_val_tensor)  # Predictions in standardized space
preds_std = preds_std.cpu().numpy()

# Inverse transform back to original physical units
preds_orig = scaler_y.inverse_transform(preds_std)  # (N,2)
y_true_orig = scaler_y.inverse_transform(y_val_tensor.cpu().numpy())  # (N,2)

# Save prediction data for scatter plots (new filename under data/)
import pickle
prediction_data = {
    'y_true': y_true_orig,
    'y_pred': preds_orig
}

results_dir = Path("data")
results_dir.mkdir(parents=True, exist_ok=True)
pred_path = results_dir / "forward_model_predictions_soft_hard.pkl"
with open(pred_path, "wb") as f:
    pickle.dump(prediction_data, f)

print(f"Soft+hard prediction data saved to: {pred_path}")
print(f"  - Validation samples: {len(y_true_orig)}")


### 5. Model Evaluation Function

This section defines the `evaluate_model` function, which is used to assess the performance of the trained model on both the training and validation datasets. The function calculates the mean squared error (MSE) and the coefficient of determination (R²) for each dataset, providing insights into the model's accuracy and generalization ability.


In [ ]:
def evaluate_model(forward_model, train_loader, val_loader, criterion, device):
    """
    Evaluate the model on both training and validation datasets.
    
    NOTE: R² must be calculated on the entire dataset, not as a batch average!
    This is because R² is a non-linear metric based on sum of squared residuals.
    """
    forward_model.eval()  # Set the model to evaluation mode

    # Collect all predictions and true values
    all_train_preds = []
    all_train_true = []
    all_val_preds = []
    all_val_true = []
    
    train_loss = 0.0
    val_loss = 0.0
    train_count = 0
    val_count = 0

    # Evaluate on training data
    with torch.no_grad():
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = forward_model(xb)
            loss = criterion(pred, yb)
            train_loss += loss.item() * xb.size(0)
            train_count += xb.size(0)
            
            # Collect predictions and true values for R² calculation
            all_train_preds.append(pred.cpu().numpy())
            all_train_true.append(yb.cpu().numpy())

    # Evaluate on validation data
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = forward_model(xb)
            loss = criterion(pred, yb)
            val_loss += loss.item() * xb.size(0)
            val_count += xb.size(0)
            
            # Collect predictions and true values for R² calculation
            all_val_preds.append(pred.cpu().numpy())
            all_val_true.append(yb.cpu().numpy())

    # Calculate average loss
    train_loss /= train_count
    val_loss /= val_count
    
    # Concatenate all batches
    all_train_preds = np.vstack(all_train_preds)
    all_train_true = np.vstack(all_train_true)
    all_val_preds = np.vstack(all_val_preds)
    all_val_true = np.vstack(all_val_true)
    
    # Calculate R² on the entire dataset (correct method)
    train_r2 = r2_score(all_train_true, all_train_preds)
    val_r2 = r2_score(all_val_true, all_val_preds)
    
    # Calculate R² for each output dimension
    train_r2_dim0 = r2_score(all_train_true[:, 0], all_train_preds[:, 0])
    train_r2_dim1 = r2_score(all_train_true[:, 1], all_train_preds[:, 1])
    val_r2_dim0 = r2_score(all_val_true[:, 0], all_val_preds[:, 0])
    val_r2_dim1 = r2_score(all_val_true[:, 1], all_val_preds[:, 1])

    print("="*70)
    print("Model Evaluation Results")
    print("="*70)
    print(f"Training Loss:   {train_loss:.4f}")
    print(f"Validation Loss: {val_loss:.4f}")
    print(f"\nOverall R²:")
    print(f"  Training R²:   {train_r2:.4f}")
    print(f"  Validation R²: {val_r2:.4f}")
    print(f"\nR² by dimension:")
    print(f"  Training   - E'_H:  {train_r2_dim0:.4f}, E''_H: {train_r2_dim1:.4f}")
    print(f"  Validation - E'_H:  {val_r2_dim0:.4f}, E''_H: {val_r2_dim1:.4f}")
    print("="*70)
    
    return train_loss, val_loss, train_r2, val_r2

# Example usage
train_loss, val_loss, train_r2, val_r2 = evaluate_model(
    forward_model, train_loader, val_loader, nn.MSELoss(), device
)


### 6. Reload training data and generate plots

**Important**: Run this section after soft+hard training finishes.

Loads from:
- `artifacts/high_stiffness/forward_model_training_history_soft_hard.csv`
- `artifacts/high_stiffness/forward_model_predictions_soft_hard.pkl`

Generates the same figures as `forward model_new.ipynb`:
1. Storage-modulus eval loss (train vs val)
2. Loss-modulus eval loss (train vs val)
3. Overall MSE (train vs val)
4. Parity scatter for $E_H'$
5. Parity scatter for $E_H''$


In [ ]:
# Load soft+hard training history from saved CSV file
import pandas as pd
import pickle
from pathlib import Path

results_dir = Path("data")
history_file = results_dir / "forward_model_training_history_soft_hard.csv"
prediction_file = results_dir / "forward_model_predictions_soft_hard.pkl"

# Check if files exist
if not history_file.exists() or not prediction_file.exists():
    print("=" * 70)
    print("ERROR: Required soft+hard result files not found!")
    print("=" * 70)
    if not history_file.exists():
        print(f"Missing: {history_file}")
    if not prediction_file.exists():
        print(f"Missing: {prediction_file}")
    print("\nThese files are generated during soft+hard forward-model training.")
    print("Please run the training cells first.")
    print("=" * 70)
    raise FileNotFoundError("Run soft+hard training cells first")

# Load training history
history_df = pd.read_csv(history_file)

# Extract data for plotting
train_eval0 = history_df["train_eval0"].tolist()
train_eval1 = history_df["train_eval1"].tolist()
val_eval0 = history_df["val_eval0"].tolist()
val_eval1 = history_df["val_eval1"].tolist()
train_eval_losses = history_df["train_eval_combined"].tolist()
val_eval_losses = history_df["val_eval_combined"].tolist()

print(f"✓ Loaded soft+hard training history: {len(history_df)} epochs")
print(f"✓ Soft+hard training completed from epoch 1 to {history_df['epoch'].max()}")


In [ ]:
# Regenerate loss curves from loaded data

import matplotlib.pyplot as plt

# Use serif + bold global settings
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif']  = ['DejaVu Serif', 'Liberation Serif']
plt.rcParams['font.weight'] = 'bold'

epochs = list(range(1, len(train_eval0) + 1))

# Storage Modulus
plt.figure(figsize=(8,8))
plt.plot(epochs, train_eval0, label="Train", linewidth=3)
plt.plot(epochs, val_eval0, label="Validation", linewidth=3)
plt.xlabel("Epoch", fontsize=28, fontweight='bold', fontfamily='serif', labelpad=10)
plt.ylabel("Loss ($E_H^{\\prime}$)",   fontsize=26, fontweight='bold', fontfamily='serif', labelpad=6)
plt.xticks(fontsize=22, fontweight='bold', fontfamily='serif')
plt.yticks(fontsize=22, fontweight='bold', fontfamily='serif')
plt.legend(prop={'size': 24, 'weight': 'bold', 'family': 'serif'})
plt.grid(False)

ax = plt.gca()
for spine in ax.spines.values():
    spine.set_linewidth(3)
plt.tick_params(axis='both', which='major', length=8, width=3)

plt.tight_layout()
plt.show()

# Loss Modulus
plt.figure(figsize=(8,8))
plt.plot(epochs, train_eval1, label="Train", linewidth=3)
plt.plot(epochs, val_eval1, label="Validation", linewidth=3)
plt.xlabel("Epoch", fontsize=28, fontweight='bold', fontfamily='serif', labelpad=10)
plt.ylabel("Loss ($E_H^{\\prime\\prime}$)",   fontsize=26, fontweight='bold', fontfamily='serif', labelpad=6)
plt.xticks(fontsize=22, fontweight='bold', fontfamily='serif')
plt.yticks(fontsize=22, fontweight='bold', fontfamily='serif')
plt.legend(prop={'size': 24, 'weight': 'bold', 'family': 'serif'})
plt.grid(False)

ax = plt.gca()
for spine in ax.spines.values():
    spine.set_linewidth(3)
plt.tick_params(axis='both', which='major', length=8, width=3)

plt.tight_layout()
plt.show()

# Overall Model
from matplotlib.ticker import FormatStrFormatter

plt.figure(figsize=(8,8))
plt.plot(epochs, train_eval_losses, label="Train", linewidth=3)
plt.plot(epochs, val_eval_losses, label="Validation", linewidth=3)
plt.xlabel("Epoch", fontsize=28, fontweight='bold', fontfamily='serif', labelpad=10)
plt.ylabel("MSE Loss",   fontsize=28, fontweight='bold', fontfamily='serif', labelpad=8)
plt.xticks(fontsize=22, fontweight='bold', fontfamily='serif')
plt.yticks(fontsize=22, fontweight='bold', fontfamily='serif')
plt.legend(prop={'size': 24, 'weight': 'bold', 'family': 'serif'})
plt.grid(False)

ax = plt.gca()
ax.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))
for spine in ax.spines.values():
    spine.set_linewidth(3)
plt.tick_params(axis='both', which='major', length=8, width=3)

plt.show()

print("Soft+hard loss curves generated from saved training history")


In [ ]:
# Load soft+hard prediction data and regenerate scatter plots
import numpy as np
from sklearn.metrics import r2_score

# Load saved soft+hard prediction data
with open(prediction_file, 'rb') as f:
    prediction_data = pickle.load(f)

y_true_orig = prediction_data['y_true']
preds_orig = prediction_data['y_pred']

print(f"Loaded soft+hard prediction data: {len(y_true_orig)} samples")

# Regenerate scatter plots
def parity_plot_with_r2_annotation(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    names=("$E_H^{\\prime}$ (MPa)", "$E_H^{\\prime\\prime}$ (MPa)"),
    point_size=55,
    alpha=0.65,
    save_prefix=None,
):
    """Larger markers + light edge for a denser look on wide soft+hard ranges."""
    colors = ['tab:blue', 'tab:orange']
    colors_line = ['tab:red', 'tab:blue']
    out_names = [
        "forward_model_parity_EpH_soft_hard.png",
        "forward_model_parity_EppH_soft_hard.png",
    ]
    rng = np.random.default_rng(42)

    for i in range(2):
        yt = y_true[:, i]
        yp = y_pred[:, i]
        r2 = r2_score(yt, yp)

        # Tiny x-jitter softens vertical strips from repeated True values within groups
        x_span = max(float(yt.max() - yt.min()), 1e-12)
        yt_plot = yt + rng.normal(0.0, 0.003 * x_span, size=len(yt))

        plt.figure(figsize=(12, 8))
        ax = plt.gca()

        ax.scatter(
            yt_plot, yp,
            c=colors[i], edgecolor='white', linewidths=0.2,
            alpha=alpha, s=point_size, zorder=2, label=names[i],
        )

        mn, mx = min(yt.min(), yp.min()), max(yt.max(), yp.max())
        pad = 0.03 * (mx - mn + 1e-12)
        ax.plot([mn - pad, mx + pad], [mn - pad, mx + pad],
                '--', color=colors_line[i], linewidth=3.5, zorder=3)
        ax.set_xlim(mn - pad, mx + pad)
        ax.set_ylim(mn - pad, mx + pad)

        ax.text(0.05, 0.95, f"R² = {r2:.3f}",
            transform=ax.transAxes,
            fontsize=28, fontweight='bold', family='serif',
            verticalalignment='top', zorder=4)

        ax.set_xlabel(f"True {names[i]}", fontsize=28, fontweight='bold', family='serif', labelpad=10)
        ax.set_ylabel(f"Pred {names[i]}", fontsize=28, fontweight='bold', family='serif', labelpad=12)

        ax.tick_params(axis='both', which='major', labelsize=24)
        for tick in ax.get_xticklabels() + ax.get_yticklabels():
            tick.set_fontweight('bold')
            tick.set_fontfamily('serif')

        for spine in ax.spines.values():
            spine.set_linewidth(3)
        plt.tick_params(axis='both', which='major', length=8, width=3)

        plt.tight_layout()
        if save_prefix is not None:
            out = Path(save_prefix) / out_names[i]
            plt.savefig(out, dpi=200, bbox_inches='tight')
            print(f"Saved {out}")
        plt.show()

parity_plot_with_r2_annotation(
    y_true_orig, preds_orig,
    point_size=55, alpha=0.65,
    save_prefix=results_dir,
)
print("Soft+hard scatter plots regenerated (larger markers)")
